# DSA Sweeper API Walkthrough

This notebook is an interactive companion to the README. It walks through the same core ideas with runnable code:

- loading the repo config
- building corner metadata from the config
- calling both public APIs with `config=` or `config_path=`
- running the one-shot export API
- comparing fixed-step and coverage-threshold export stop behavior
- inspecting the exported timestep records
- creating an incremental observe-and-plan session
- replaying several incremental updates as an interactive video

Most of the notebook logic now lives in `src/notebooks/api_showcase.py` so the notebook can stay focused on explanation and visualization.


## 1. Notebook Setup

This cell finds the repo root, keeps `src` importable, and loads the helper module that owns the heavier notebook logic.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src").exists() and (candidate / "config.yaml").exists():
            return candidate
    raise RuntimeError("Could not find the DSA Sweeper repo root from the current notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.notebooks import api_showcase as showcase

print(f"Repo root: {REPO_ROOT}")

## 2. Load The Config And Build Corner Metadata

The helper module loads the repo config, resolves the image path, and builds the `tl` / `tr` / `br` / `bl` corner mapping for plotting.

Both public APIs can now bootstrap directly from that same config object, or load it internally with `config_path=` if you prefer.

In [ ]:
context = showcase.build_showcase_context(REPO_ROOT)
showcase.show_payload("Loaded Config Summary", showcase.build_config_summary(context))
showcase.show_payload("Config-backed API call patterns", showcase.build_api_call_examples(context))

### Visualize The Search Map

This large preview makes the input image easy to spot before we call either API. The heatmap is drawn in geographic coordinates, and the corner labels make the coverage area obvious.

In [ ]:
showcase.plot_search_map(context)

## 3. One-Shot Export API

This is the README's offline simulation flow. The helper module now calls the export API with `config=context.config`, so image path, source ppm, and geo corners all come from the loaded config by default.

If you would rather let the API load the YAML itself, the same pattern works with `config_path="config.yaml"`.

The one-shot export API supports three stopping patterns:

- `max_steps=<int>` for a fixed simulation horizon
- `max_steps=None` to run until the planner finishes
- `stop_when_covered_percent=<0..100>` to stop once that percentage of the initial heatmap value has been cleared

To keep the notebook fast, we first run a capped export to inspect the record format, then compare it with a low-threshold coverage-stop demo.


In [ ]:
EXPORT_MAX_STEPS = 100

exports = showcase.run_export_demo(context, max_steps=EXPORT_MAX_STEPS)
showcase.show_payload(
    "Step-capped export window",
    showcase.summarize_export_window(exports),
)
list(exports)


### Compare Export Stop Policies

The next cell contrasts a fixed `max_steps` export with a coverage-threshold export that uses `max_steps=None`.

For notebook speed, the coverage threshold is intentionally small here. The exact same call pattern scales directly to `stop_when_covered_percent=95.0` in Python and `make export-paths OUTPUT=exports MAX_STEPS=none STOP_WHEN_COVERED_PERCENT=95` from the terminal.


In [ ]:
COVERAGE_STOP_PERCENT = showcase.DEFAULT_EXPORT_COVERAGE_STOP_PERCENT

coverage_stop_exports = showcase.run_export_demo(
    context,
    max_steps=None,
    stop_when_covered_percent=COVERAGE_STOP_PERCENT,
)

showcase.show_payload(
    "One-shot export stop behavior",
    {
        "step_capped_example": {
            "call": (
                "export_drone_geojsons_from_image("
                f"config=context.config, max_steps={EXPORT_MAX_STEPS})"
            ),
            **showcase.summarize_export_window(exports),
        },
        "coverage_stop_example": {
            "call": (
                "export_drone_geojsons_from_image("
                "config=context.config, "
                f"max_steps=None, stop_when_covered_percent={COVERAGE_STOP_PERCENT})"
            ),
            **showcase.summarize_export_window(coverage_stop_exports),
        },
    },
)


### Inspect The Exported Record Shape

Each drone export is a JSON array with one timestep record per simulated step. The next cell shows the first and last record for the first configured drone ID, plus the timestamp spacing.

In [ ]:
first_drone_id = showcase.first_drone_id(exports)
first_drone_records = exports[first_drone_id]

print(f"{first_drone_id} record count: {len(first_drone_records)}")
print(f"first timestamp: {first_drone_records[0]['timestamp']}")
print(f"last timestamp:  {first_drone_records[-1]['timestamp']}")

if len(first_drone_records) > 1:
    delta_t = first_drone_records[1]["timestamp"] - first_drone_records[0]["timestamp"]
    print(f"timestamp spacing: {delta_t}")

showcase.show_payload(f"First {first_drone_id} record", first_drone_records[0])
showcase.show_payload(f"Last {first_drone_id} record", first_drone_records[-1])

### Quick Export Summary Across All Drones

This cell gives a compact overview of the exported arrays without dumping every record.

In [ ]:
export_summary = showcase.summarize_exports(exports)
showcase.show_payload("Per-drone export summary", export_summary)

### Visualize The Exported Flight Paths

These plots are intentionally large and heavily labeled. Solid tracks show drone motion, squares mark the starting positions, X markers show the final positions, and the second panel makes the camera ground projections obvious.

In [ ]:
showcase.plot_export_paths(context, exports)

## 4. Incremental Session API

This is the README's integration flow. Another system owns motion and sensing, while this repo owns planning state.

Below, we create a session with `config=context.config`, inspect the first plan, then feed synthetic observations back into the planner using the current state as an example input.

The same session bootstrap also works with `config_path="config.yaml"`.

In [ ]:
session = showcase.create_demo_session(context, step_seconds=0.2)
initial_plan = session.get_plan()
showcase.show_payload("Session planner settings", initial_plan["planner"])

### Build Example Observations

A real external system would send live position and projection-point measurements. For demo purposes, we reuse the session's current state and camera projection point so the API call stays fully reproducible.

In [ ]:
observations = showcase.build_observations_from_plan(initial_plan)
first_drone_id = showcase.first_drone_id(observations)
showcase.show_payload(f"Example observation payload for {first_drone_id}", observations[first_drone_id])

In [ ]:
step_result = session.observe_and_plan(observations)

showcase.show_payload("Returned planner settings after observe_and_plan", step_result["planner"])
showcase.show_payload(f"{first_drone_id} result", step_result["drones"][first_drone_id])

### Visualize The Session Recommendations

This figure turns the incremental API response into something you can read at a glance. Circles are the current drone states, solid arrows point to next flight targets, dashed arrows point to camera targets, and X markers show the current projection points.

In [ ]:
showcase.plot_session_recommendations(context, step_result)

### Replay The Incremental Session As A Video

This cell runs the incremental API for many updates and renders the result as a notebook video. The built-in player underneath the animation gives you play, pause, rewind, and frame-by-frame stepping controls while the planner reacts to each new observation.

In [ ]:
replay = showcase.build_incremental_replay(
    context,
    step_seconds=float(initial_plan["planner"]["step_seconds"]),
    max_steps=24,
)
print(showcase.replay_summary_text(replay))
replay_animation = showcase.render_incremental_replay(context, replay)

### Optional: Override `dt_seconds` Per Update

The session's configured `step_seconds` is the default. A caller can still provide a different `dt_seconds` value for any individual `observe_and_plan(...)` call.

The planner settings below stay the same because the session configuration is unchanged; only the decay scaling for that single update call changes.

In [ ]:
step_result_fast = session.observe_and_plan(observations, dt_seconds=0.05)
showcase.show_payload("Planner settings after a per-call dt_seconds override", step_result_fast["planner"])

## 5. Ideas For Further Testing

Some easy next experiments:

- change `max_steps` in the one-shot export cell and inspect how timestamps grow
- set `max_steps=None` and compare the record counts with a step-capped run
- raise or lower `stop_when_covered_percent` and see how quickly the export stops
- change `step_seconds` and see how motion spacing and timestamps change
- lower `search_decay_percent_per_100ms` to make the heatmap fade more gradually
- swap between `config=` and `config_path=` to match how your own integration will load settings
- pass different `initial_altitudes_agl` values to compare camera projection behavior
- replace the synthetic observations with real measurements from an external system

If you want a script instead of a notebook, the same code patterns map directly to the README examples and the CLI commands.
